# 天気図パターン分類 - すぐに使う (v14)

1. まず下の「セットアップ」セルを1回実行してください(コードは折りたたまれています)。
2. その後、一番下の入力フォームで条件を選んで実行してください。

> セットアップセル実行後に表示される `バージョン: vX` が、上のタイトルの版と
> 一致していれば最新版が動いています。古い場合はランタイムを再接続し、
> ページを再読み込みしてください。

In [ ]:
#@title 🔧 セットアップ(最初に1回だけ実行してください) { display-mode: "form" }
NOTEBOOK_VERSION = "v14"

REPO_URL = "https://github.com/awg-yk/weather-pattern-classification.git"
# mainはコードのみ(天気図の画像データを含まない)ので数秒でcloneできる。
# 画像データを含むブランチはcloneすると3GB超になり事実上終わらないため使わない。
# **まだmainに入っていない機能を試すときは、ここに作業ブランチ名を入れる。**
# 例: claude/detection-plan-validation-t7sohl(注釈方式の重みとテンプレート)
BRANCH = "main" #@param {type:"string"}
REPO_DIR = "/content/weather-pattern-classification"
WEIGHTS_PATH = f"{REPO_DIR}/weights/model.pt"
# 検出した高低気圧の枠を描き込んだ画像で学習した重み。入力が違うので別名。
# **必ず --annotate と組にすること。**素の天気図を渡すと、モデルは見たことの
# ない絵を受け取ることになり、成績が静かに落ちる。
ANNOT_WEIGHTS_PATH = f"{REPO_DIR}/weights/model_annot.pt"
TEMPLATES_DIR = f"{REPO_DIR}/data/templates"
MARKS_DIR = f"{REPO_DIR}/data/marks"

import subprocess, os, sys, shutil, time

# 再実行時にカレントディレクトリがREPO_DIR配下(前回の%cdの結果)になっている場合、
# 次のrmtreeでカレントディレクトリ自体を削除してしまい、その後のgit cloneが
# 「Unable to read current working directory」で失敗する。それを防ぐため、
# rmtree前に必ず安全なディレクトリへ退避しておく。
os.chdir("/content")

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

os.environ["GIT_TERMINAL_PROMPT"] = "0"  # 認証プロンプト待ちで固まるのを防ぐ
print("リポジトリ取得中...")
t0 = time.time()
clone = subprocess.run(
    ["timeout", "120", "git", "clone", "--depth", "1", "--single-branch",
     "-b", BRANCH, REPO_URL, REPO_DIR],
    check=False,
    capture_output=True,
    text=True,
)
if clone.returncode == 124:
    raise TimeoutError(
        "git cloneが120秒経っても終わりませんでした。\n"
        "ランタイムを再接続してから、もう一度実行してください。"
    )
elif clone.returncode != 0:
    raise RuntimeError(
        f"git cloneに失敗しました(終了コード: {clone.returncode})\n"
        f"--- git stderr ---\n{clone.stderr}"
    )
print(f"取得完了 ({time.time() - t0:.1f}秒)")

%cd {REPO_DIR}
# Colabにはtorch/torchvision/numpy/pandas/matplotlib/requests/tqdm/pillowが
# 標準搭載済みなので、追加で必要なpdf2image・scipyだけ入れる。
!pip install -q pdf2image scipy
!apt-get -qq install -y fonts-noto-cjk poppler-utils

assert os.path.exists(WEIGHTS_PATH), f"モデルの重みが見つかりません: {WEIGHTS_PATH}"

# scripts.*/src.* はこのノートブックを1回のカーネルセッション内で複数回セットアップ
# し直すと、ディスク上のファイルはgit cloneで最新化されてもPythonの
# sys.modulesキャッシュが古いままになり、修正が反映されない(バージョン表示だけ
# 新しくなって中身は古いまま、という紛らわしい状態になる)。
# 毎回確実に最新コードを読み込むよう、importの前にキャッシュを破棄しておく。
for mod_name in list(sys.modules):
    if mod_name == "scripts" or mod_name.startswith("scripts.") or mod_name == "src" or mod_name.startswith("src."):
        del sys.modules[mod_name]

sys.path.append(REPO_DIR)
import matplotlib.pyplot as plt
from src.labels import LABEL_JA
from scripts.gradcam import explain_predictions_above_threshold
from scripts.fetch_and_predict import fetch_chart, chart_exists
from google.colab import files
import numpy as np
from PIL import Image
from scripts.preprocess_jma import DEFAULT_STAMP_BOX, autocrop_to_content, mask_stamp_box


def annotation_available():
    """注釈方式が使える状態か(重みとテンプレートが揃っているか)を返す。"""
    missing = [p for p in (ANNOT_WEIGHTS_PATH, TEMPLATES_DIR) if not os.path.exists(p)]
    return (not missing), missing


def make_annotated(image_path: str) -> str:
    """検出した枠を描き込んだ画像を作り、そのパスを返す。

    **描き方は学習に使ったものと揃える。**同梱の重みは枠のみ(前線の縁取り
    なし)で作った画像で学習してあるので、ここも枠のみにする。
    """
    from scripts.annotate_charts import annotate_one

    image = Image.open(image_path).convert("RGB")
    image = autocrop_to_content(image)
    image = mask_stamp_box(image, DEFAULT_STAMP_BOX)
    marked, detections = annotate_one(
        np.array(image), TEMPLATES_DIR,
        MARKS_DIR if os.path.exists(MARKS_DIR) else None,
        boxes=True, fronts=False,
    )
    out_path = "/content/annotated.png"
    Image.fromarray(marked).save(out_path)
    print(f"検出: 高気圧 {len(detections.highs)}個 / 低気圧 {len(detections.lows)}個"
          f"(中心が枠外の系 {len(detections.edge_highs) + len(detections.edge_lows)}個)")
    return out_path


def classify_and_show(image_path: str, threshold: float, annotate: bool = False):
    """画像1枚を分類し、確信度がthresholdを超えたラベル分だけヒートマップを表示、
    それ以外はテキストのみで確信度一覧を出す。

    annotate=True にすると、先に高低気圧を検出して枠を描き込み、注釈付き画像で
    学習した重みを使う。**Grad-CAMは「モデルがどこを見たか」しか示さないが、
    枠は「検出が当たったか」を示す。**別のことを示すので、両方あると読み解ける。
    """
    weights = WEIGHTS_PATH
    if annotate:
        ok, missing = annotation_available()
        if not ok:
            print("注釈方式は使えません(見つからないもの: "
                  + ", ".join(os.path.basename(m) for m in missing) + ")")
            print("素の天気図の方式で続けます。")
            annotate = False
        else:
            image_path = make_annotated(image_path)
            weights = ANNOT_WEIGHTS_PATH

    display_image, overlays, ranked = explain_predictions_above_threshold(
        image_path=image_path,
        weights_path=weights,
        threshold=threshold,
        # 描き込み済みの画像には前処理を二重にかけない
        apply_preprocess=not annotate,
    )

    n_panels = len(overlays) + 1
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))
    if n_panels == 1:
        axes = [axes]
    axes[0].imshow(display_image)
    axes[0].set_title("検出結果(枠つき)" if annotate else "入力画像(前処理後)")
    axes[0].axis("off")

    for ax, (label, prob, overlay) in zip(axes[1:], overlays):
        ax.imshow(overlay)
        ax.set_title(f"{LABEL_JA[label]}\n({prob * 100:.1f}%)")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    if not overlays:
        print(f"確信度{threshold * 100:.0f}%を超えるラベルはありませんでした。\n")

    print("--- 全ラベルの確信度 ---")
    for label, prob in ranked:
        print(f"{LABEL_JA[label]}: {prob * 100:.1f}%")


print(f"セットアップ完了 | バージョン: {NOTEBOOK_VERSION} | ブランチ: {BRANCH}")

# 注釈方式が使えるかどうかを、ここで先に知らせる。使えないまま
# USE_ANNOTATION にチェックを入れると素の方式に落ちるだけなので、
# 「チェックしたのに何も変わらない」と見えてしまう。
_ok, _missing = annotation_available()
if _ok:
    print("注釈方式: 使えます(USE_ANNOTATION にチェックを入れてください)")
else:
    print("注釈方式: 使えません。足りないもの: "
          + ", ".join(os.path.basename(m) for m in _missing))
    print(f"  -> このブランチ({BRANCH})には入っていません。"
          "上の BRANCH に、重みとテンプレートを含むブランチ名を入れて実行し直してください。")

In [ ]:
#@title 画像を用意して分類 { display-mode: "form" }
#@markdown **MODE**: `upload`=画像をアップロード / `date`=日付指定で取得(2000-01-01〜2022-09-30は手動アーカイブ、2022-10-01以降は気象庁JSMAPアーカイブから自動取得)
MODE = "date" #@param ["upload", "date"]

#@markdown **日付**(MODE="date"のときのみ使用)
YEAR = 2025 #@param [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026] {type:"raw"}
MONTH = 1 #@param [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] {type:"raw"}
DAY = 1 #@param [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31] {type:"raw"}
HOUR = 0 #@param [0, 12] {type:"raw"}

#@markdown **表示するラベルの確信度しきい値**(これを超えたラベルだけヒートマップ画像で表示)
THRESHOLD = 0.5 #@param {type:"slider", min:0.0, max:1.0, step:0.05}

#@markdown **検出結果を描き込んでから分類する**(高低気圧を先に見つけて枠を描く方式)
#@markdown <br>左端の画像に枠が出るので、**検出が当たっているかを目で確かめられます**。
USE_ANNOTATION = False #@param {type:"boolean"}

if MODE == "upload":
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]
elif MODE == "date":
    import datetime
    from scripts.fetch_and_predict import EARLIEST_KNOWN_DATE
    from scripts.fetch_manual_chart import MANUAL_ARCHIVE_START_DATE
    target = datetime.date(YEAR, MONTH, DAY)

    if not chart_exists(target, HOUR):
        raise FileNotFoundError(
            f"{target.isoformat()} {HOUR}Z の天気図が見つかりません。"
            f"対応範囲は{MANUAL_ARCHIVE_START_DATE.isoformat()}以降(0Zまたは12Zのみ)です。"
        )

    date_str = f"{YEAR:04d}-{MONTH:02d}-{DAY:02d}"
    image_path = str(fetch_chart(date_str, hour=HOUR))
    print("取得:", image_path)
else:
    raise ValueError('MODEは "upload" か "date" を指定してください')

classify_and_show(image_path, threshold=THRESHOLD, annotate=USE_ANNOTATION)